# Calendar vs calendar + lags

Train **LightGBM** on OPSD German hourly renewables (2015–2019 slice): compare **calendar-only** vs **calendar + lag/rolling** features. See [docs/methodology.md](../docs/methodology.md).

**Prerequisite:** cached OPSD parquet or env `ENERGY_FP_OPSD_PATH` (first CSV download is large).

**LightGBM** (Light Gradient Boosting Machine) is a high-performance, open-source gradient boosting framework developed by Microsoft that specializes in efficient, tree-based learning algorithms. It is designed for speed, scalability, and handling large-scale datasets, often used for ranking, classification, and regression tasks


In [ ]:
from __future__ import annotations

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from energy_features import (
    load_opsd_generation,
    make_calendar_features,
    make_lag_rolling_block,
)
from lightgbm import (
    LGBMRegressor,
)  # highly efficient, open-source gradient boosting framework developed by Microsoft.
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
TZ = "Europe/Berlin"
TEST_DAYS = 90
RNG_SEED = 42

df = load_opsd_generation()
df = df.sort_index()
if df.index.tz is None:
    df.index = df.index.tz_localize("UTC")
df = df.tz_convert(TZ)
df = df.loc["2015":"2019"].dropna(how="any")
df.head()

In [ ]:
def smape(
    y_true: np.ndarray, y_pred: np.ndarray
) -> float:  # Symmetric Mean Absolute Percentage Error
    num = np.abs(y_true - y_pred)
    denom = np.abs(y_true) + np.abs(y_pred)
    mask = denom > 1e-9
    return float(np.mean(2.0 * num[mask] / denom[mask]))


def train_eval(col: str) -> pd.Series:
    y = df[col].astype(float)
    X_cal = make_calendar_features(y.index, country="DE")
    X_full = X_cal.join(
        make_lag_rolling_block(
            y.rename(col),
            lags=[24, 48, 168],
            windows=[24, 168],
            prefix=col,
        )
    ).dropna()
    y_fit = y.loc[X_full.index]

    split_time = y_fit.index.max() - pd.Timedelta(days=TEST_DAYS)
    train_mask = y_fit.index <= split_time
    Xt_tr, Xt_te = (
        X_full.loc[train_mask],
        X_full.loc[~train_mask],
    )  # the full set of features used in the machine learning model
    yt_tr, yt_te = y_fit.loc[train_mask], y_fit.loc[~train_mask]

    X_cal_tr = X_cal.loc[X_full.index].loc[
        train_mask
    ]  #  represents the training set for the calendar-based features.
    X_cal_te = X_cal.loc[X_full.index].loc[~train_mask]

    common_kw = dict(
        objective="regression",
        random_state=RNG_SEED,
        n_estimators=400,
        learning_rate=0.05,
        max_depth=-1,
        subsample=0.8,
        colsample_bytree=0.8,
    )

    m_cal = LGBMRegressor(**common_kw)
    m_cal.fit(X_cal_tr, yt_tr)

    m_full = LGBMRegressor(**common_kw)
    m_full.fit(Xt_tr, yt_tr)

    pred_cal = m_cal.predict(X_cal_te)
    pred_full = m_full.predict(Xt_te)

    yt = yt_te.to_numpy()
    mae_cal = mean_absolute_error(yt, pred_cal)
    mae_full = mean_absolute_error(yt, pred_full)
    rmse_cal = float(np.sqrt(mean_squared_error(yt, pred_cal)))
    rmse_full = float(np.sqrt(mean_squared_error(yt, pred_full)))
    sm_cal = smape(yt, pred_cal)
    sm_full = smape(yt, pred_full)

    lift_mae = (mae_cal - mae_full) / mae_cal if mae_cal else np.nan

    return pd.Series(
        {
            "MAE_cal": mae_cal,
            "MAE_cal_lag": mae_full,
            "lift_MAE_%": 100.0 * lift_mae,
            "RMSE_cal": rmse_cal,
            "RMSE_cal_lag": rmse_full,
            "sMAPE_cal_%": sm_cal,
            "sMAPE_cal_lag_%": sm_full,
            "n_train": len(yt_tr),
            "n_test": len(yt_te),
        },
        name=col,
    )

In [ ]:
rows = [train_eval(c) for c in df.columns]
summary = pd.DataFrame(rows)
summary

In [ ]:
# Residual ACF on first technology (example diagnostic)
col = df.columns[0]
y = df[col].astype(float)
X_full = (
    make_calendar_features(y.index, country="DE")
    .join(make_lag_rolling_block(y.rename(col), [24, 48, 168], [24, 168], prefix=col))
    .dropna()
)
y_fit = y.loc[X_full.index]
split_time = y_fit.index.max() - pd.Timedelta(days=TEST_DAYS)
train_mask = y_fit.index <= split_time
Xt_tr, Xt_te = X_full.loc[train_mask], X_full.loc[~train_mask]
yt_tr, yt_te = y_fit.loc[train_mask], y_fit.loc[~train_mask]
model = LGBMRegressor(
    objective="regression",
    random_state=RNG_SEED,
    n_estimators=400,
    learning_rate=0.05,
)
model.fit(Xt_tr, yt_tr)
resid = (yt_te - model.predict(Xt_te)).values
max_lag = 72
acf = [float(np.corrcoef(resid[k:], resid[:-k])[0, 1]) if k > 0 else 1.0 for k in range(max_lag)]
plt.figure(figsize=(9, 3))
plt.stem(range(max_lag), acf, basefmt=" ")
plt.title(f"Residual ACF ({col}) — calendar+lags model")
plt.xlabel("Lag (h)")
plt.ylabel("ACF")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()